In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


def encontrar_raiz_repo(inicio=None):
    """Sube por el árbol de directorios hasta encontrar la raíz del repositorio."""
    actual = (inicio or Path.cwd()).resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / ".git").exists():
            return carpeta
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. "
        "Ejecutá el notebook desde dentro de la copia local del repo."
    )


RAIZ = encontrar_raiz_repo()
DIR_PROCESSED = RAIZ / "data" / "processed"

RUTA_INMUEBLES = DIR_PROCESSED / "mercadolibre_caba_procesado.csv"
RUTA_SUBTE = DIR_PROCESSED / "subte_limpio.csv"
RUTA_HOSPITALES = DIR_PROCESSED / "hospitales_limpio.csv"
RUTA_DELITOS = DIR_PROCESSED / "delitos_2023_2025_unificado_limpio.csv"
RUTA_COLECTIVOS = DIR_PROCESSED / "colectivos_limpio.csv"
RUTA_SALIDA = DIR_PROCESSED / "dataframefinal.csv"

INSUMOS = [RUTA_INMUEBLES, RUTA_SUBTE, RUTA_HOSPITALES, RUTA_DELITOS, RUTA_COLECTIVOS]
faltantes = [ruta.name for ruta in INSUMOS if not ruta.exists()]

print("Raíz del repositorio:", RAIZ)
if faltantes:
    print("\nFaltan insumos en data/processed/:")
    for nombre in faltantes:
        print("  -", nombre)
else:
    print("Todos los insumos están disponibles.")

Raíz del repositorio: /Users/valentinzuppa/Desktop/TP-Analisis-Descriptivo-Inmobiliario-CABA
Todos los insumos están disponibles.


In [2]:
df = pd.read_csv(RUTA_INMUEBLES, low_memory=False)
df.head(5)

,Fuente,Item_ID,Tipo_Operacion,Tipo_Alquiler,Tipo_Propiedad,Subtipo_Propiedad,Titulo,Tipo_Vendedor,Precio_Raw,Precio,...,Flag_Operacion_Anomala,Flag_Tipo_Propiedad_Anomalo,Flag_Superficie_Anomala,Flag_Ambientes_Anomalo,Flag_Coordenadas_Anomalas,Flag_Expensas_Anomalas,Flag_Amenities_Anomalos,Flag_Faltantes_Criticos,Flag_Anomalia_General,Motivo_Anomalia
0,Mercado Libre,MLA3224379444,Venta,NaN,Departamento,NaN,Se Vende Un Departamento De 3 Ambientes Al Fre...,Inmobiliaria,US$ 100.000,100000.0,...,0,0,0,0,0,0,0,0,0,Sin anomalías detectadas
1,Mercado Libre,MLA3558686250,Venta,NaN,Departamento,NaN,Venta Departamento - 4 Ambientes - Palermo,Inmobiliaria,US$ 179.000,179000.0,...,0,0,0,0,0,0,0,0,0,Sin anomalías detectadas
2,Mercado Libre,MLA1945483967,Venta,NaN,Departamento,NaN,Venta Departamento 3 Amb Desarrollo Villa Urquiza,Inmobiliaria,US$ 315.000,315000.0,...,0,0,0,0,0,1,1,0,1,Expensas cero sospechosas | Conflicto ML/texto...
3,Mercado Libre,MLA1945603777,Venta,NaN,Departamento,Penthhouse,Penthouse Dúplex Con Jardín Y Vista Al Río - N...,Inmobiliaria,US$ 870.000,870000.0,...,0,0,0,0,0,0,0,0,0,Sin anomalías detectadas
4,Mercado Libre,MLA3183809008,Venta,NaN,Departamento,NaN,Departamento - Caballito,Inmobiliaria,US$ 353.352,353352.0,...,0,0,0,0,0,1,1,0,1,Expensas cero sospechosas | Conflicto ML/texto...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65846 entries, 0 to 65845
Data columns (total 67 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Fuente                       65846 non-null  str    
 1   Item_ID                      65846 non-null  str    
 2   Tipo_Operacion               65846 non-null  str    
 3   Tipo_Alquiler                9293 non-null   str    
 4   Tipo_Propiedad               65846 non-null  str    
 5   Subtipo_Propiedad            15575 non-null  str    
 6   Titulo                       65846 non-null  str    
 7   Tipo_Vendedor                65826 non-null  str    
 8   Precio_Raw                   65846 non-null  str    
 9   Precio                       65846 non-null  float64
 10  Moneda                       65846 non-null  str    
 11  Expensas_Raw                 61740 non-null  str    
 12  Expensas                     61740 non-null  float64
 13  Moneda_Expensas            

In [4]:
print(df.shape)
nulos = df.isnull().sum()
nulos[nulos > 0]

(65846, 67)


Tipo_Alquiler                56553
Subtipo_Propiedad            50271
Tipo_Vendedor                   20
Expensas_Raw                  4106
Expensas                      4106
Moneda_Expensas               4106
Direccion_Publicada             71
Calle                         1295
Altura                        1295
Barrio_Publicado               133
Latitud                         73
Longitud                        73
Piso                         37369
Ambientes                     8509
Dormitorios                  10635
Baños                         2829
Cocheras                     13554
Bauleras                     23540
Superficie_Total_m2            245
Superficie_Cubierta_m2        1897
Superficie_No_Cubierta_m2     2645
Superficie_Balcon_m2         46262
Antiguedad                    1933
Cantidad_Pisos_Edificio      33564
Departamentos_Por_Piso       41865
Disposicion                  18329
Orientacion                  30077
dtype: int64

In [5]:
df = df[df["Tipo_Propiedad"].isin(["Departamento", "PH", "Casa"])].copy()
print("Filas en alcance residencial:", df.shape[0])

Filas en alcance residencial: 55582


Normalizacion e imputación de barrios

In [6]:
MAPA_BARRIOS = {
    "Once": "Balvanera",
    "Barrio Norte": "Recoleta",
    "Las Cañitas": "Palermo",
    "Paternal": "La Paternal",
    "Santa Rita": "Villa Santa Rita",
    "Villa Gral. Mitre": "Villa General Mitre",
    "Velez Sarsfield": "Vélez Sarsfield",
    "Gral.": "Villa Santa Rita",
    "Avda.": "Villa Santa Rita",
}

df["Barrio_Normalizado"] = df["Barrio_Publicado"].replace(MAPA_BARRIOS)

BARRIOS_CABA = {
    "Agronomía", "Almagro", "Balvanera", "Barracas", "Belgrano", "Boedo", "Caballito",
    "Chacarita", "Coghlan", "Colegiales", "Constitución", "Flores", "Floresta", "La Boca",
    "La Paternal", "Liniers", "Mataderos", "Monserrat", "Monte Castro", "Nueva Pompeya",
    "Núñez", "Palermo", "Parque Avellaneda", "Parque Chacabuco", "Parque Chas",
    "Parque Patricios", "Puerto Madero", "Recoleta", "Retiro", "Saavedra", "San Cristóbal",
    "San Nicolás", "San Telmo", "Vélez Sarsfield", "Versalles", "Villa Crespo",
    "Villa del Parque", "Villa Devoto", "Villa General Mitre", "Villa Lugano", "Villa Luro",
    "Villa Ortúzar", "Villa Pueyrredón", "Villa Real", "Villa Riachuelo", "Villa Santa Rita",
    "Villa Soldati", "Villa Urquiza",
}

no_reconocidos = set(df["Barrio_Normalizado"].dropna().unique()) - BARRIOS_CABA
assert not no_reconocidos, f"Quedaron barrios sin mapear: {no_reconocidos}"

print("Barrios sin dato tras el mapeo (NaN, se excluyen de KPIs por zona):",
      df["Barrio_Normalizado"].isna().sum())

Barrios sin dato tras el mapeo (NaN, se excluyen de KPIs por zona): 133


Limpieza Moneda

In [7]:
TIPO_CAMBIO_USD = 1522.0  #dólar blue, promedio 14-17/08/2026

df["Precio_USD"] = np.where(df["Moneda"] == "ARS", df["Precio"] / TIPO_CAMBIO_USD, df["Precio"])
df["Expensas_USD"] = np.where(
    df["Moneda_Expensas"].fillna("ARS") == "ARS", df["Expensas"] / TIPO_CAMBIO_USD, df["Expensas"]
)

Limpieza Antiguedad


In [8]:
df["Antiguedad_limpia"] = df["Antiguedad"].where(df["Antiguedad"].between(0, 150))

Filtro de calidad de precio para KPIS monetarios

In [9]:
#Nos aseguramos que para hacer los claculos de kpis con precios los mismos pasen las flags de la base y un filtro propio (que estén dentro del percentil 0,01 y 0,99 para su categoría)

mask_flags_ok = (
    (df["Flag_Precio_Anomalo"] == 0)
    & (df["Flag_Operacion_Anomala"] == 0)
    & (df["Flag_Superficie_Anomala"] == 0)
    & (df["Superficie_Total_m2"] > 0)
)

df["Precio_m2_USD"] = np.where(mask_flags_ok, df["Precio_USD"] / df["Superficie_Total_m2"], np.nan)

limites_m2 = (
    df[mask_flags_ok]
    .groupby("Tipo_Operacion")["Precio_m2_USD"]
    .quantile([0.01, 0.99])
    .unstack()
    .rename(columns={0.01: "p1_m2", 0.99: "p99_m2"})
)
df = df.join(limites_m2, on="Tipo_Operacion")

mask_precio_ok = mask_flags_ok & df["Precio_m2_USD"].between(df["p1_m2"], df["p99_m2"])
print("Filas que pasan el filtro de calidad de precio:", mask_precio_ok.sum(), "/", mask_flags_ok.sum())

Filas que pasan el filtro de calidad de precio: 53628 / 54720


Limepiza ambientes

In [10]:
#Agrupamos todo lo que tenga 4 ambientes o + para que tenes un volumen mas acorde
def bucket_ambientes(x):
    if pd.isna(x) or x <= 0:
        return np.nan
    if x <= 3:
        return str(int(x))
    return "4+"

df["Ambientes_Agrupado"] = df["Ambientes"].apply(bucket_ambientes)

KPI - Tasa de confort

In [11]:
AMENITIES_CONFORT = [
    "Amoblado", "Ascensor", "Balcon", "Terraza", "Patio", "Pileta", "Parrilla", "SUM",
    "Gimnasio", "Laundry", "Seguridad", "Aire_Acondicionado", "Calefaccion", "Lavadero",
    "Dormitorio_Suite",
]
df["Tasa_Confort_Amenities"] = df[AMENITIES_CONFORT].sum(axis=1) / len(AMENITIES_CONFORT)

print(df["Tasa_Confort_Amenities"].describe())

count    55582.000000
mean         0.252667
std          0.180773
min          0.000000
25%          0.133333
50%          0.200000
75%          0.333333
max          1.000000
Name: Tasa_Confort_Amenities, dtype: float64


KPI - Precio por m2

In [12]:
kpi_precio_m2_barrio = (
    df[mask_precio_ok & df["Barrio_Normalizado"].notna()]
    .groupby(["Barrio_Normalizado", "Tipo_Operacion"])["Precio_m2_USD"]
    .agg(Precio_m2_Promedio="mean", Precio_m2_Mediana="median", N="count")
    .round(1)
)
print(kpi_precio_m2_barrio.sort_values("N", ascending=False).head(10))

#Desagregado por tipo de propiedad
kpi_precio_m2_barrio_tipo = (
    df[mask_precio_ok & df["Barrio_Normalizado"].notna()]
    .groupby(["Barrio_Normalizado", "Tipo_Propiedad", "Tipo_Operacion"])["Precio_m2_USD"]
    .agg(Precio_m2_Promedio="mean", Precio_m2_Mediana="median", N="count")
    .round(1)
)

                                   Precio_m2_Promedio  Precio_m2_Mediana     N
Barrio_Normalizado Tipo_Operacion                                             
Villa Urquiza      Venta                       2457.2             2477.8  2487
Caballito          Venta                       2373.9             2282.8  2390
Flores             Venta                       1758.4             1727.3  2299
Almagro            Venta                       2114.2             2075.5  2267
Villa Crespo       Venta                       2351.8             2409.7  2211
Palermo            Venta                       3079.9             2950.0  2197
Balvanera          Venta                       1690.1             1588.8  2155
Belgrano           Venta                       3236.7             3109.6  2151
Núñez              Venta                       3304.3             3181.8  2115
Villa Devoto       Venta                       2372.1             2337.7  2091


KPIS - Rentabilidad Bruta, Neta y Payback Period

In [13]:
GRANO = ["Barrio_Normalizado", "Tipo_Propiedad", "Ambientes_Agrupado"]
MIN_N = 5  # mínimo de avisos por combinación para confiar en la mediana

base_mask = mask_precio_ok & df["Barrio_Normalizado"].notna() & df["Ambientes_Agrupado"].notna()

venta = df[base_mask & (df["Tipo_Operacion"] == "Venta")]
alquiler = df[base_mask & (df["Tipo_Operacion"] == "Alquiler")]

venta_g = venta.groupby(GRANO)["Precio_USD"].agg(["median", "count"])
venta_g = venta_g[venta_g["count"] >= MIN_N]["median"].rename("Precio_Venta_Prom_USD")

alquiler_g = alquiler.groupby(GRANO)["Precio_USD"].agg(["median", "count"])
alquiler_g = alquiler_g[alquiler_g["count"] >= MIN_N]["median"].rename("Alquiler_Mensual_Prom_USD")

In [14]:
#Se calculas solamente en las expensas donde no hay anomalías; ya que el dataset cuenta con las mismas en un 42,78% de los registros

expensas_mask = (
    (df["Flag_Expensas_Anomalas"] == 0)
    & df["Expensas_USD"].notna()
    & df["Barrio_Normalizado"].notna()
    & df["Ambientes_Agrupado"].notna()
)
expensas_g = df[expensas_mask].groupby(GRANO)["Expensas_USD"].median().rename("Expensas_Prom_USD")

kpi_zonas = pd.concat([venta_g, alquiler_g, expensas_g], axis=1)
kpi_zonas = kpi_zonas.dropna(subset=["Precio_Venta_Prom_USD", "Alquiler_Mensual_Prom_USD"])
print("Combinaciones Barrio x Tipo x Ambientes con venta Y alquiler suficientes:", len(kpi_zonas))

Combinaciones Barrio x Tipo x Ambientes con venta Y alquiler suficientes: 139


In [15]:
#Rentabilidad Bruta Estimada
kpi_zonas["Rentabilidad_Bruta_Estimada"] = (
    kpi_zonas["Alquiler_Mensual_Prom_USD"] * 12
) / kpi_zonas["Precio_Venta_Prom_USD"]

In [16]:
#Rentabilidad Neta Estimada
MANTENIMIENTO_ANUAL_PCT = 0.01
kpi_zonas["Mantenimiento_Est_Anual_USD"] = kpi_zonas["Precio_Venta_Prom_USD"] * MANTENIMIENTO_ANUAL_PCT

kpi_zonas["Rentabilidad_Neta_Estimada"] = (
    (kpi_zonas["Alquiler_Mensual_Prom_USD"] * 12)
    - (kpi_zonas["Expensas_Prom_USD"] * 12)
    - kpi_zonas["Mantenimiento_Est_Anual_USD"]
) / kpi_zonas["Precio_Venta_Prom_USD"]

In [17]:
#Payback Period
kpi_zonas["Payback_Period_Anios"] = 1 / kpi_zonas["Rentabilidad_Neta_Estimada"]

print(kpi_zonas.sort_values("Rentabilidad_Bruta_Estimada", ascending=False).head(10)[
    ["Precio_Venta_Prom_USD", "Alquiler_Mensual_Prom_USD",
     "Rentabilidad_Bruta_Estimada", "Rentabilidad_Neta_Estimada", "Payback_Period_Anios"]
])

                                                      Precio_Venta_Prom_USD  \
Barrio_Normalizado Tipo_Propiedad Ambientes_Agrupado                          
Núñez              Departamento   4+                               505000.0   
Monserrat          Departamento   3                                100000.0   
Barracas           PH             2                                 57500.0   
San Telmo          PH             2                                 80000.0   
Villa Devoto       Casa           4+                               450000.0   
Retiro             Departamento   2                                 90000.0   
Colegiales         Departamento   4+                               417973.0   
Villa Lugano       Departamento   3                                 57900.0   
La Boca            Departamento   2                                 59000.0   
Caballito          Casa           4+                               416000.0   

                                                   

Indice de spbre/subvaluacion

In [18]:
df["Antiguedad_Rango"] = pd.cut(
    df["Antiguedad_limpia"], bins=[-0.1, 10, 30, 50, 150], labels=["0-10", "10-30", "30-50", "50+"]
)
df["Confort_Rango"] = pd.cut(
    df["Tasa_Confort_Amenities"], bins=[-0.1, 0.2, 0.4, 1.01], labels=["Bajo", "Medio", "Alto"]
)

model_mask = mask_precio_ok & df["Barrio_Normalizado"].notna() & df["Antiguedad_limpia"].notna()

df["Precio_Predicho_USD"] = np.nan
df["Indice_Subvaluacion"] = np.nan

NIVELES_DE_GRUPO = [
    ["Barrio_Normalizado", "Tipo_Propiedad", "Antiguedad_Rango", "Confort_Rango"],
    ["Barrio_Normalizado", "Tipo_Propiedad", "Antiguedad_Rango"],
    ["Barrio_Normalizado", "Tipo_Propiedad"],
    ["Barrio_Normalizado"],
]
MIN_N_GRUPO = 8  # mínimo de avisos comparables para confiar en la mediana del grupo

for operacion in ["Venta", "Alquiler"]:
    sub = df[model_mask & (df["Tipo_Operacion"] == operacion)].copy()

    precio_m2_predicho = pd.Series(np.nan, index=sub.index)
    for columnas_grupo in NIVELES_DE_GRUPO:
        pendientes = precio_m2_predicho.isna()
        if not pendientes.any():
            break
        conteos = sub.groupby(columnas_grupo, observed=True)["Precio_m2_USD"].transform("count")
        medianas = sub.groupby(columnas_grupo, observed=True)["Precio_m2_USD"].transform("median")
        usar_este_nivel = pendientes & (conteos >= MIN_N_GRUPO)
        precio_m2_predicho[usar_este_nivel] = medianas[usar_este_nivel]

    precio_predicho = precio_m2_predicho * sub["Superficie_Total_m2"]
    df.loc[sub.index, "Precio_Predicho_USD"] = precio_predicho
    df.loc[sub.index, "Indice_Subvaluacion"] = (sub["Precio_USD"] - precio_predicho) / precio_predicho

    print(f"{operacion}: n = {len(sub)}  |  sin grupo comparable = {precio_predicho.isna().sum()}")

print(df["Indice_Subvaluacion"].describe())

Venta: n = 47127  |  sin grupo comparable = 3
Alquiler: n = 4591  |  sin grupo comparable = 42
count    51673.000000
mean         0.023994
std          0.257445
min         -0.815213
25%         -0.131000
50%          0.000000
75%          0.146025
max          4.733939
Name: Indice_Subvaluacion, dtype: float64


Score de oportunidad

In [19]:
neta_por_zona = kpi_zonas["Rentabilidad_Neta_Estimada"].rename("Rentabilidad_Neta_Zona")
df = df.join(neta_por_zona, on=GRANO)

venta_mask = model_mask & (df["Tipo_Operacion"] == "Venta") & df["Rentabilidad_Neta_Zona"].notna()
score_df = df[venta_mask].copy()

def norm01(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

score_df["Rentabilidad_Neta_norm"] = norm01(score_df["Rentabilidad_Neta_Zona"])
score_df["Subvaluacion_norm"] = norm01(-score_df["Indice_Subvaluacion"])  #negativo = oportunidad

def factor_confianza(row):
    if row["Flag_Faltantes_Criticos"] == 1 or row["Flag_Coordenadas_Anomalas"] == 1:
        return 0.0  # se descarta del ranking
    if row["Flag_Anomalia_General"] == 0:
        return 1.0  # sin anomalías
    return 0.6  # anomalía no crítica

score_df["Factor_Confianza"] = score_df.apply(factor_confianza, axis=1)
score_df["Score_Oportunidad"] = (
    0.5 * score_df["Rentabilidad_Neta_norm"] + 0.5 * score_df["Subvaluacion_norm"]
) * score_df["Factor_Confianza"] * 100

ranking_oportunidades = (
    score_df[score_df["Factor_Confianza"] > 0]
    .sort_values("Score_Oportunidad", ascending=False)
)

print(ranking_oportunidades[
    ["Barrio_Normalizado", "Tipo_Propiedad", "Ambientes", "Precio_USD",
     "Indice_Subvaluacion", "Rentabilidad_Neta_Zona", "Factor_Confianza", "Score_Oportunidad"]
].head(15))

      Barrio_Normalizado Tipo_Propiedad  Ambientes  Precio_USD  \
30861              Núñez   Departamento        4.0    395000.0   
30790              Núñez   Departamento        4.0    210000.0   
30794              Núñez   Departamento        4.0    210000.0   
30468              Núñez   Departamento        4.0    245000.0   
30467              Núñez   Departamento        4.0    245000.0   
29470              Núñez   Departamento        6.0    449000.0   
29427              Núñez   Departamento        4.0    278000.0   
29931              Núñez   Departamento        6.0    550000.0   
13877              Núñez   Departamento        4.0    344000.0   
30796              Núñez   Departamento        4.0    210000.0   
13011              Núñez   Departamento        4.0    410000.0   
30016              Núñez   Departamento        4.0    267900.0   
29677              Núñez   Departamento        4.0    150000.0   
11383              Núñez   Departamento        4.0    220000.0   
30509     

Nueva columna de cantidad de metros a la estacion de subte más cercana

In [20]:
subtes = pd.read_csv(RUTA_SUBTE)
subtes.head(5)

,ID,Tipo_Transporte,Nombre,Linea,Direccion,Barrio,Comuna,Latitud,Longitud,Flag_Coordenada_Anomala,Flag_Duplicado
0,1,Subte,Caseros,H,No informado,No informado,No informado,-34.635749,-58.398930,0,0
1,2,Subte,Inclan - Mezquita Al Ahmad,H,No informado,No informado,No informado,-34.629374,-58.400972,0,0
2,3,Subte,Humberto 1°,H,No informado,No informado,No informado,-34.623091,-58.402325,0,0
3,4,Subte,Venezuela,H,No informado,No informado,No informado,-34.615241,-58.404734,0,0
4,5,Subte,Once - 30 De Diciembre,H,No informado,No informado,No informado,-34.608934,-58.406039,0,0


In [26]:
print(subtes.shape)

(90, 11)


In [32]:
from scipy.spatial import cKDTree

R_TIERRA = 6_371_000  # metros

def latlon_a_xyz(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = R_TIERRA * np.cos(lat_rad) * np.cos(lon_rad)
    y = R_TIERRA * np.cos(lat_rad) * np.sin(lon_rad)
    z = R_TIERRA * np.sin(lat_rad)
    return np.column_stack([x, y, z])

subte_xyz = latlon_a_xyz(subtes["Latitud"].values, subtes["Longitud"].values)
tree = cKDTree(subte_xyz)

tiene_coords = df["Latitud"].notna() & df["Longitud"].notna()
df["Distancia_Subte_m"] = np.nan

props_xyz = latlon_a_xyz(
    df.loc[tiene_coords, "Latitud"].values,
    df.loc[tiene_coords, "Longitud"].values,
)
distancias, _ = tree.query(props_xyz, k=1)
df.loc[tiene_coords, "Distancia_Subte_m"] = distancias

print(df[["Titulo", "Barrio_Publicado", "Distancia_Subte_m"]].head(10))

                                              Titulo Barrio_Publicado  \
0  Se Vende Un Departamento De 3 Ambientes Al Fre...        Balvanera   
1         Venta Departamento - 4 Ambientes - Palermo          Palermo   
2  Venta Departamento 3 Amb Desarrollo Villa Urquiza    Villa Urquiza   
3  Penthouse Dúplex Con Jardín Y Vista Al Río - N...            Núñez   
4                           Departamento - Caballito        Caballito   
5  Departamento En Venta Recoleta Ayacucho - Melo...         Recoleta   
6  A Estrenar, Últimas Unidades En Excelente Torr...          Almagro   
7  Departamento En Venta De Pozo 5 Ambientes En B...         Belgrano   
8  ¡deptos De 4 Y 5 Ambientes Con Balcones Terraz...     Villa Crespo   
9  Departamento En Flores De 4 Amb - Entr Abril 2...           Flores   

   Distancia_Subte_m  
0         304.078622  
1         266.353839  
2         339.079085  
3        1056.343258  
4        1152.177752  
5         548.306767  
6         257.134479  
7         34

In [33]:
vacios = df["Distancia_Subte_m"].isna().sum()
print(f"Cantidad de valores vacíos: {vacios}")

Cantidad de valores vacíos: 63


**Cantidad de metros al hospital más cercano**

In [34]:
hospitales = pd.read_csv(RUTA_HOSPITALES)

# Descartar hospitales sin coordenadas válidas, si los hubiera
hospitales_ok = hospitales.dropna(subset=["latitud", "longitud"]).copy()

hosp_xyz = latlon_a_xyz(hospitales_ok["latitud"].values, hospitales_ok["longitud"].values)
tree_hosp = cKDTree(hosp_xyz)

tiene_coords = df["Latitud"].notna() & df["Longitud"].notna()
df["Distancia_Hospital_m"] = np.nan

props_xyz = latlon_a_xyz(
    df.loc[tiene_coords, "Latitud"].values,
    df.loc[tiene_coords, "Longitud"].values,
)
distancias, _ = tree_hosp.query(props_xyz, k=1)
df.loc[tiene_coords, "Distancia_Hospital_m"] = distancias

print(df[["Titulo", "Barrio_Publicado", "Distancia_Subte_m", "Distancia_Hospital_m"]].head(10))

                                              Titulo Barrio_Publicado  \
0  Se Vende Un Departamento De 3 Ambientes Al Fre...        Balvanera   
1         Venta Departamento - 4 Ambientes - Palermo          Palermo   
2  Venta Departamento 3 Amb Desarrollo Villa Urquiza    Villa Urquiza   
3  Penthouse Dúplex Con Jardín Y Vista Al Río - N...            Núñez   
4                           Departamento - Caballito        Caballito   
5  Departamento En Venta Recoleta Ayacucho - Melo...         Recoleta   
6  A Estrenar, Últimas Unidades En Excelente Torr...          Almagro   
7  Departamento En Venta De Pozo 5 Ambientes En B...         Belgrano   
8  ¡deptos De 4 Y 5 Ambientes Con Balcones Terraz...     Villa Crespo   
9  Departamento En Flores De 4 Amb - Entr Abril 2...           Flores   

   Distancia_Subte_m  Distancia_Hospital_m  
0         304.078622           1819.182241  
1         266.353839            774.573978  
2         339.079085           1911.976246  
3        1056.34

**Cantidad de delitos en un radio de 1000 metros**

In [35]:
delitos = pd.read_csv(RUTA_DELITOS, low_memory=False)
print(delitos.shape)

"""Nueva columna de cantidad de delitos en un radio de 1000 metros"""

delito_xyz = latlon_a_xyz(delitos["Latitud"].values, delitos["Longitud"].values)
tree_delito = cKDTree(delito_xyz)

tiene_coords = df["Latitud"].notna() & df["Longitud"].notna()
df["Cantidad_Delitos_1000m"] = np.nan

props_xyz = latlon_a_xyz(
    df.loc[tiene_coords, "Latitud"].values,
    df.loc[tiene_coords, "Longitud"].values,
)
cantidades = tree_delito.query_ball_point(props_xyz, r=1000, return_length=True)
df.loc[tiene_coords, "Cantidad_Delitos_1000m"] = cantidades

print(df[["Titulo", "Barrio_Publicado", "Cantidad_Delitos_1000m"]].head(10))

vacios = df["Cantidad_Delitos_1000m"].isna().sum()
print(f"Cantidad de valores vacíos: {vacios}")



(438081, 16)
                                              Titulo Barrio_Publicado  \
0  Se Vende Un Departamento De 3 Ambientes Al Fre...        Balvanera   
1         Venta Departamento - 4 Ambientes - Palermo          Palermo   
2  Venta Departamento 3 Amb Desarrollo Villa Urquiza    Villa Urquiza   
3  Penthouse Dúplex Con Jardín Y Vista Al Río - N...            Núñez   
4                           Departamento - Caballito        Caballito   
5  Departamento En Venta Recoleta Ayacucho - Melo...         Recoleta   
6  A Estrenar, Últimas Unidades En Excelente Torr...          Almagro   
7  Departamento En Venta De Pozo 5 Ambientes En B...         Belgrano   
8  ¡deptos De 4 Y 5 Ambientes Con Balcones Terraz...     Villa Crespo   
9  Departamento En Flores De 4 Amb - Entr Abril 2...           Flores   

   Cantidad_Delitos_1000m  
0                 20036.0  
1                 14637.0  
2                  5742.0  
3                  5738.0  
4                  8542.0  
5              

**Cantidad de colectivos en un radio de 500 metros**

In [38]:
colectivos = pd.read_csv(RUTA_COLECTIVOS)

print(colectivos.shape)

"""Nueva columna de cantidad de paradas de colectivo en un radio de 500 metros"""

# Descartar paradas con coordenada marcada como anómala
colectivos_ok = colectivos.loc[colectivos["Flag_Coordenada_Anomala"] == 0].copy()

colectivo_xyz = latlon_a_xyz(
    colectivos_ok["Latitud"].values, colectivos_ok["Longitud"].values
)
tree_colectivo = cKDTree(colectivo_xyz)

tiene_coords = df["Latitud"].notna() & df["Longitud"].notna()
df["Cantidad_Colectivos_500m"] = np.nan

props_xyz = latlon_a_xyz(
    df.loc[tiene_coords, "Latitud"].values,
    df.loc[tiene_coords, "Longitud"].values,
)
cantidades = tree_colectivo.query_ball_point(props_xyz, r=500, return_length=True)
df.loc[tiene_coords, "Cantidad_Colectivos_500m"] = cantidades

print(df[["Titulo", "Barrio_Publicado", "Cantidad_Colectivos_500m"]].head(10))

vacios = df["Cantidad_Colectivos_500m"].isna().sum()
print(f"Cantidad de valores vacíos: {vacios}")

(6962, 13)
                                              Titulo Barrio_Publicado  \
0  Se Vende Un Departamento De 3 Ambientes Al Fre...        Balvanera   
1         Venta Departamento - 4 Ambientes - Palermo          Palermo   
2  Venta Departamento 3 Amb Desarrollo Villa Urquiza    Villa Urquiza   
3  Penthouse Dúplex Con Jardín Y Vista Al Río - N...            Núñez   
4                           Departamento - Caballito        Caballito   
5  Departamento En Venta Recoleta Ayacucho - Melo...         Recoleta   
6  A Estrenar, Últimas Unidades En Excelente Torr...          Almagro   
7  Departamento En Venta De Pozo 5 Ambientes En B...         Belgrano   
8  ¡deptos De 4 Y 5 Ambientes Con Balcones Terraz...     Villa Crespo   
9  Departamento En Flores De 4 Amb - Entr Abril 2...           Flores   

   Cantidad_Colectivos_500m  
0                      75.0  
1                      59.0  
2                      39.0  
3                      18.0  
4                      38.0  
5    

In [39]:
df.to_csv(RUTA_SALIDA, index=False)
print("Dataset final guardado en:", RUTA_SALIDA)
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

Dataset final guardado en: /Users/valentinzuppa/Desktop/TP-Analisis-Descriptivo-Inmobiliario-CABA/data/processed/dataframefinal.csv
Filas: 55582 | Columnas: 85
